In [1]:
# NB01_data_preparation.ipynb

# %% [markdown]
# # NB01 – Data Preparation
# **Purpose:** Load the Polish Companies Bankruptcy dataset (.arff files),
# merge all five yearly files, clean and impute missing values, and save
# a master parquet file for downstream notebooks.
#
# **Download data from:**
# https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data
#
# Place 1year.arff ~ 5year.arff in `data/raw/` before running.
#
# **Output:** `data/processed/polish_master.parquet`

# %%
import os
import warnings
import numpy as np
import pandas as pd
from scipy.io import arff

warnings.filterwarnings("ignore")

RAW_DIR  = "../data/raw/"
PROC_DIR = "../data/processed/"
os.makedirs(RAW_DIR,  exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

print("Directories ready.")

# %% [markdown]
# ## 1. Load .arff files

# %%
ARFF_FILES = {
    f"{i}year": os.path.join(RAW_DIR, f"{i}year.arff")
    for i in range(1, 6)
}

frames = []
for year_key, fpath in ARFF_FILES.items():
    if not os.path.exists(fpath):
        print(f"[SKIP] {fpath} not found.")
        continue
    data, meta = arff.loadarff(fpath)
    df = pd.DataFrame(data)
    # Decode byte-string columns produced by arff loader
    df = df.applymap(lambda x: x.decode() if isinstance(x, bytes) else x)
    df["year_horizon"] = int(year_key[0])   # 1 … 5
    frames.append(df)
    print(f"Loaded {year_key}: {len(df):,} rows, {df.shape[1]} cols")

if not frames:
    raise FileNotFoundError(
        "No .arff files found. Download from UCI and place in data/raw/"
    )

master = pd.concat(frames, ignore_index=True)
print(f"\nMaster shape: {master.shape}")

# %% [markdown]
# ## 2. Synthetic fallback (remove once real data is available)
# Generates a structurally identical dataset so all downstream notebooks
# can run end-to-end without the raw files.

# %%
# Comment out this cell after placing real .arff files in data/raw/
if "master" not in dir() or len(master) == 0:
    np.random.seed(42)
    N = 15_000

    attr_cols = {f"Attr{i}": np.random.randn(N) for i in range(1, 65)}
    master = pd.DataFrame(attr_cols)

    # Inject realistic distributions for key ratios
    master["Attr1"]  = np.clip(np.random.beta(2, 3, N),        0.01, 0.99)   # net profit / total assets
    master["Attr5"]  = np.clip(np.random.lognormal(0, 0.6, N), 0.10, 20.0)  # current ratio
    master["Attr8"]  = np.clip(np.random.normal(0.5, 0.3, N),  -1.0,  3.0)  # EBIT / total assets
    master["Attr21"] = np.clip(np.random.lognormal(0, 1.0, N), 0.05, 10.0)  # sales / total assets
    master["Attr31"] = np.clip(np.random.beta(3, 2, N),        0.01, 0.99)  # debt ratio

    # Bankruptcy label: ~5 % positive rate
    logit = (
        -2
        - 3.0 * master["Attr1"]
        - 1.5 * master["Attr5"]
        - 2.0 * master["Attr8"]
        + 1.5 * master["Attr31"]
        + np.random.randn(N) * 0.8
    )
    master["class"]        = (1 / (1 + np.exp(-logit)) > 0.95).astype(int).astype(str)
    master["year_horizon"] = np.random.choice([1, 2, 3, 4, 5], N)

    print(f"[SYNTHETIC] Generated {N:,} rows for demo.")
    print(master["class"].value_counts())

# %% [markdown]
# ## 3. Target variable & column renaming

# %%
# Standardise target column across all year files
master["default"] = (
    master["class"].astype(str).str.strip().isin(["1", "1.0", "b'1'"])
).astype(int)

# Rename first 10 attributes to interpretable names
# (based on dataset documentation)
rename_map = {
    "Attr1" : "net_profit_to_assets",
    "Attr2" : "total_liabilities_to_assets",
    "Attr3" : "working_capital_to_assets",
    "Attr4" : "current_assets_to_short_liabilities",
    "Attr5" : "cash_to_current_liabilities",
    "Attr6" : "retained_earnings_to_assets",
    "Attr7" : "ebit_to_assets",
    "Attr8" : "book_value_equity_to_liabilities",
    "Attr9" : "sales_to_assets",
    "Attr10": "equity_to_assets",
}
master.rename(columns=rename_map, inplace=True)

print("Renamed columns sample:")
print([c for c in master.columns if not c.startswith("Attr")][:15])

# %% [markdown]
# ## 4. Outlier clipping (1st / 99th percentile)

# %%
numeric_cols = master.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ("default", "year_horizon")]

for col in numeric_cols:
    lo, hi = master[col].quantile([0.01, 0.99])
    master[col] = master[col].clip(lo, hi)

print(f"Clipped {len(numeric_cols)} numeric columns.")

# %% [markdown]
# ## 5. Missing value imputation

# %%
missing     = master.isnull().sum()
missing_pct = (missing / len(master) * 100).round(2)
missing_report = pd.DataFrame({"missing_n": missing, "missing_pct": missing_pct})
missing_report = (
    missing_report[missing_report["missing_n"] > 0]
    .sort_values("missing_pct", ascending=False)
)

print(f"Columns with missing values: {len(missing_report)}")
display(missing_report.head(20))

# Median imputation
for col in numeric_cols:
    if master[col].isnull().any():
        master[col].fillna(master[col].median(), inplace=True)

print(f"\nRemaining missing values: {master.isnull().sum().sum()}")

# %% [markdown]
# ## 6. Save

# %%
out_path = os.path.join(PROC_DIR, "polish_master.parquet")
master.to_parquet(out_path, index=False)

print(f"Saved : {out_path}")
print(f"Shape : {master.shape}")
print(f"Default rate : {master['default'].mean():.4f}  "
      f"({master['default'].mean()*100:.2f}%)")
print(master[["default", "year_horizon"]].value_counts().sort_index())

Directories ready.
Loaded 1year: 7,027 rows, 66 cols
Loaded 2year: 10,173 rows, 66 cols
Loaded 3year: 10,503 rows, 66 cols
Loaded 4year: 9,792 rows, 66 cols
Loaded 5year: 5,910 rows, 66 cols

Master shape: (43405, 66)
Renamed columns sample:
['net_profit_to_assets', 'total_liabilities_to_assets', 'working_capital_to_assets', 'current_assets_to_short_liabilities', 'cash_to_current_liabilities', 'retained_earnings_to_assets', 'ebit_to_assets', 'book_value_equity_to_liabilities', 'sales_to_assets', 'equity_to_assets', 'class', 'year_horizon', 'default']
Clipped 64 numeric columns.
Columns with missing values: 64


,missing_n,missing_pct
Attr37,18984,43.74
Attr21,5854,13.49
Attr27,2764,6.37
Attr60,2152,4.96
Attr45,2147,4.95
Attr24,922,2.12
Attr54,812,1.87
Attr28,812,1.87
Attr64,812,1.87
Attr53,812,1.87



Remaining missing values: 0
Saved : ../data/processed/polish_master.parquet
Shape : (43405, 67)
Default rate : 0.0482  (4.82%)
default  year_horizon
0        1                6756
         2                9773
         3               10008
         4                9277
         5                5500
1        1                 271
         2                 400
         3                 495
         4                 515
         5                 410
Name: count, dtype: int64
